# Microproyecto 01

## Clasificación de MRI con Redes Neuronales Convolucionales para Tipos de Cáncer

Materia: Técnicas de Deep Learning
Programa: Maestría en IA
Estudiantes: Sergio Pardo, Jefferson Moreno, John Pino
Fecha: Abril 2026



## 1. Contexto del problema

La resonancia magnética (MRI) permite apoyar la detección y clasificación de diferentes tipos de cáncer cerebral. En este microproyecto se trabajará con imágenes bidimensionales individuales extraídas de resonancias, en lugar de procesar volúmenes completos, con el fin de reducir el costo computacional y abordar el problema como una tarea de clasificación de imágenes.

Las clases definidas son:

- glioma
- meningioma
- pituitary
- healthy

Se evalúan dos enfoques de transferencia de aprendizaje sobre el mismo dataset y la misma partición de datos:

- EfficientNet-B0 como arquitectura eficiente de referencia.
- ResNet50 como arquitectura residual, incluyendo una fase adicional de fine-tuning.



## 2. Objetivo

Desarrollar métodos basados en redes neuronales convolucionales que permitan clasificar con exactitud imágenes extraídas de MRIs en una de las cuatro categorías planteadas en el microproyecto, comparando dos arquitecturas preentrenadas bajo un pipeline común de preparación de datos.



## 3. Configuración inicial



In [ ]:

import hashlib
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "dataset"
classes = sorted([folder for folder in DATASET_DIR.iterdir() if folder.is_dir()])
class_names = [folder.name for folder in classes]

RANDOM_SEED = 42
SAMPLES_PER_CLASS = 4
IMAGE_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
RESNET_FINE_TUNE_LR = 1e-5
NUM_EPOCHS = 10
NUM_FINE_TUNE_EPOCHS = 10
AUTOTUNE = tf.data.AUTOTUNE

BEST_EFFICIENTNET_PATH = BASE_DIR / "best_efficientnet_b0.keras"
BEST_RESNET_PATH = BASE_DIR / "best_resnet50.weights.h5"
BEST_RESNET_FT_PATH = BASE_DIR / "best_resnet50_fine_tuning.weights.h5"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Directorio base: {BASE_DIR}")
print(f"Dataset: {DATASET_DIR}")
print(f"Clases: {class_names}")
print(f"Dispositivos disponibles: {tf.config.list_physical_devices()}")




## 4. Dataset



In [ ]:

class_counts = {
    folder.name: len([path for path in folder.iterdir() if path.is_file()])
    for folder in sorted(DATASET_DIR.iterdir())
    if folder.is_dir()
}
print(f"Total de imágenes: {sum(class_counts.values())}")
class_counts




## 5. EDA

En esta sección se realiza una exploración rápida del dataset para validar tres aspectos básicos antes del modelado: distribución por clase, consistencia general de tamaños y modos de color, y ejemplos visuales aleatorios de cada categoría.



In [ ]:

image_records = []
for class_dir in classes:
    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file():
            continue
        with Image.open(image_path) as image:
            width, height = image.size
            image_records.append(
                {
                    "class": class_dir.name,
                    "path": image_path,
                    "width": width,
                    "height": height,
                    "mode": image.mode,
                }
            )

class_counts = Counter(record["class"] for record in image_records)
size_counts = Counter((record["width"], record["height"]) for record in image_records)
mode_counts = Counter(record["mode"] for record in image_records)

print("Resumen del dataset")
print(f"- Total de imágenes: {len(image_records)}")
print(f"- Tamaños más frecuentes: {size_counts.most_common(5)}")
print(f"- Total de tamaños distintos: {len(size_counts)}")
print(f"- Modos de color detectados: {dict(mode_counts)}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(class_counts.keys(), class_counts.values(), color=["#355070", "#6d597a", "#b56576", "#e56b6f"])
ax.set_title("Distribución de imágenes por clase")
ax.set_ylabel("Cantidad")
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

rng = random.Random(RANDOM_SEED)
fig, axes = plt.subplots(len(classes), SAMPLES_PER_CLASS, figsize=(12, 10))

for i, class_dir in enumerate(classes):
    image_files = [path for path in sorted(class_dir.iterdir()) if path.is_file()]
    sampled_paths = rng.sample(image_files, k=SAMPLES_PER_CLASS)

    for j, image_path in enumerate(sampled_paths):
        ax = axes[i, j]
        ax.axis("off")

        with Image.open(image_path) as image:
            ax.imshow(image, cmap="gray")

        if j == 0:
            ax.set_title(class_dir.name)

fig.suptitle("Muestra aleatoria por clase", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()




### 5.1 Conclusiones preliminares

El dataset presenta una distribución relativamente balanceada entre las categorías, lo cual es favorable para el entrenamiento del modelo. No obstante, las imágenes tienen 387 resoluciones distintas, por lo que será necesario aplicar un preprocesamiento que permita estandarizar su tamaño antes de ingresarlas a la red neuronal.



## 6. Preprocesamiento y preparación del pipeline

En esta sección se construye el pipeline de datos que se utilizará en el microproyecto. Para ello, se realiza la limpieza básica de archivos, la eliminación de duplicados exactos, el preprocesamiento de las imágenes y la preparación de los conjuntos de entrenamiento, validación y prueba.

Aspectos considerados:

- Limpieza de archivos no válidos.
- Eliminación de imágenes duplicadas exactas.
- Redimensionamiento de imágenes.
- Conversión al formato requerido por el modelo.
- Normalización.
- División del dataset en entrenamiento, validación y prueba.

La partición de datos se define una sola vez y se reutiliza en los dos enfoques, para hacer comparables los resultados.



In [ ]:

# Esta validación permite descartar archivos propios del sistema operativo y formatos no esperados.
def is_valid_image(path):
    path = Path(path)
    return path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"} and not path.name.startswith(".")


def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = tf.cast(image, tf.float32)
    return image, label


def build_dataset(paths, labels, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(paths), seed=RANDOM_SEED)

    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset


valid_files = [
    path
    for path in DATASET_DIR.rglob("*")
    if path.is_file() and is_valid_image(path)
]

unique_files = []
seen_hashes = set()

for path in valid_files:
    file_hash = hashlib.md5(path.read_bytes()).hexdigest()
    if file_hash in seen_hashes:
        continue
    seen_hashes.add(file_hash)
    unique_files.append(path)

removed_duplicates = len(valid_files) - len(unique_files)

class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
all_paths = [str(path) for path in unique_files]
all_targets = [class_to_idx[Path(path).parent.name] for path in unique_files]

# Obtenemos el set para train.
train_paths, temp_paths, train_targets, temp_targets = train_test_split(
    all_paths,
    all_targets,
    test_size=0.2,
    stratify=all_targets,
    random_state=RANDOM_SEED,
)

# Obtenenemos los sets para validación y prueba.
val_paths, test_paths, val_targets, test_targets = train_test_split(
    temp_paths,
    temp_targets,
    test_size=0.5,
    stratify=temp_targets,
    random_state=RANDOM_SEED,
)

train_dataset = build_dataset(train_paths, train_targets, shuffle=True)
val_dataset = build_dataset(val_paths, val_targets)
test_dataset = build_dataset(test_paths, test_targets)


def summarize_split(name, labels):
    distribution = Counter(class_names[label] for label in labels)
    print(f"- {name}: {len(labels)} imágenes | {dict(distribution)}")


print("Resumen del pipeline")
print(f"- Imágenes válidas: {len(valid_files)}")
print(f"- Duplicados exactos removidos: {removed_duplicates}")
print(f"- Imágenes finales: {len(unique_files)}")
print(f"- Total verificado: {len(train_paths) + len(val_paths) + len(test_paths)}")
print("\nDistribución por split")
summarize_split("Train", train_targets)
summarize_split("Validation", val_targets)
summarize_split("Test", test_targets)

batch_images, batch_labels = next(iter(train_dataset))
print("\nValidación de batch")
print(f"- Batch shape: {tuple(batch_images.shape)}")
print(f"- Clases en batch: {sorted(tf.unique(batch_labels).y.numpy().tolist())}")




## 7. Diseño de la arquitectura del modelo

Para este microproyecto se utilizarán dos modelos convolucionales preentrenados. Ambos se adaptan a una salida de 4 clases, correspondiente a las categorías del dataset.



### 7.1 EfficientNet-B0

Para este microproyecto se utilizará EfficientNet-B0 con pesos preentrenados en ImageNet. Se elige este enfoque porque permite aprovechar características visuales ya aprendidas y reduce la complejidad frente al diseño de una CNN desde cero.

La arquitectura se adaptará a una salida de 4 clases, correspondiente a las categorías del dataset. El modelo se dejará preparado para que, en la etapa de entrenamiento, se congele inicialmente el extractor de características y se entrene únicamente la capa clasificadora final.

En esta sección se define el modelo base, la capa de salida y la función de pérdida.



In [ ]:

efficientnet_data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomContrast(0.2),
    ],
    name="efficientnet_data_augmentation",
)

efficientnet_base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    pooling="avg",
)

# Se congela el extractor para entrenar inicialmente solo la capa clasificadora.
efficientnet_base_model.trainable = False

efficientnet_inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = efficientnet_data_augmentation(efficientnet_inputs)
efficientnet_features = efficientnet_base_model(x, training=False)
efficientnet_outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(efficientnet_features)

efficientnet_model = tf.keras.Model(efficientnet_inputs, efficientnet_outputs, name="efficientnet_b0_mri")

efficientnet_criterion = tf.keras.losses.SparseCategoricalCrossentropy()
efficientnet_metric_name = "accuracy"

print("Configuración del modelo")
print(f"- Modelo: EfficientNet-B0")
print(f"- Salida final: {len(class_names)} clases")
print(f"- Input shape: {(IMAGE_SIZE, IMAGE_SIZE, 3)}")
print(f"- Función de pérdida: {efficientnet_criterion.name}")
print(f"- Métrica principal: {efficientnet_metric_name}")




### 7.2 ResNet50

El segundo enfoque utiliza ResNet50, una arquitectura residual preentrenada en ImageNet. La primera fase congela el extractor de características y entrena únicamente la cabeza clasificadora; posteriormente se realiza fine-tuning de los bloques superiores.



In [ ]:

resnet_data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomContrast(0.2),
    ],
    name="resnet_data_augmentation",
)

resnet_base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    pooling="avg",
)

resnet_base_model.trainable = False

resnet_inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = resnet_data_augmentation(resnet_inputs)
x = tf.keras.layers.Lambda(
    tf.keras.applications.resnet50.preprocess_input,
    name="resnet50_preprocess",
)(x)
x = resnet_base_model(x, training=False)
x = tf.keras.layers.Dropout(0.2)(x)
resnet_outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

resnet_model = tf.keras.Model(resnet_inputs, resnet_outputs, name="resnet50_mri")
resnet_criterion = tf.keras.losses.SparseCategoricalCrossentropy()

print("Configuración del modelo")
print("- Modelo: ResNet50")
print(f"- Salida final: {len(class_names)} clases")
print(f"- Input shape: {(IMAGE_SIZE, IMAGE_SIZE, 3)}")
print(f"- Función de pérdida: {resnet_criterion.name}")
print(f"- Parámetros totales: {resnet_model.count_params():,}")
print(f"- Parámetros entrenables: {sum(np.prod(variable.shape) for variable in resnet_model.trainable_variables):,}")




## 8. Entrenamiento del modelo

En esta sección se entrenan los modelos utilizando transferencia de aprendizaje. Para mantener el enfoque del taller, en la primera fase se congela el extractor de características y se actualiza únicamente la cabeza clasificadora final.



### 8.1 Entrenamiento de EfficientNet-B0

Decisiones de entrenamiento:

- Número de épocas: 10.
- Tamaño de lote: 32.
- Tasa de aprendizaje: 1e-3.
- Implementación: model.compile(...) y model.fit(...).
- Criterio de selección: se guarda el modelo con menor pérdida de validación.



In [ ]:

efficientnet_optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
efficientnet_model.compile(
    optimizer=efficientnet_optimizer,
    loss=efficientnet_criterion,
    metrics=[efficientnet_metric_name],
)

efficientnet_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=BEST_EFFICIENTNET_PATH,
    monitor="val_loss",
    save_best_only=True,
)

print("Configuración de entrenamiento")
print(f"- Épocas: {NUM_EPOCHS}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Learning rate: {LEARNING_RATE}")
print(f"- Optimizador: {efficientnet_optimizer.__class__.__name__}")

efficientnet_history = efficientnet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=NUM_EPOCHS,
    callbacks=[efficientnet_checkpoint_callback],
    verbose=1,
)

efficientnet_model = tf.keras.models.load_model(BEST_EFFICIENTNET_PATH)
efficientnet_best_val_loss = min(efficientnet_history.history["val_loss"])
efficientnet_epochs = range(1, len(efficientnet_history.history["loss"]) + 1)

print(f"\nMejor pérdida de validación: {efficientnet_best_val_loss:.4f}")
print(f"Modelo guardado en: {BEST_EFFICIENTNET_PATH}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(efficientnet_epochs, efficientnet_history.history["loss"], marker="o", label="Train")
axes[0].plot(efficientnet_epochs, efficientnet_history.history["val_loss"], marker="o", label="Validation")
axes[0].set_title("Pérdida por época")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(efficientnet_epochs, efficientnet_history.history["accuracy"], marker="o", label="Train")
axes[1].plot(efficientnet_epochs, efficientnet_history.history["val_accuracy"], marker="o", label="Validation")
axes[1].set_title("Accuracy por época")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()




#### 8.1.1 Conclusiones preliminares

Las curvas de EfficientNet-B0 muestran un entrenamiento estable. La pérdida de entrenamiento disminuyó de 0.6621 a 0.2315, mientras que la exactitud de entrenamiento aumentó de 0.7550 a 0.9183. En validación, el modelo alcanzó su mejor pérdida en la época 9, con val_loss = 0.2510 y val_accuracy = 0.8970.

La diferencia entre entrenamiento y validación es moderada y no evidencia un sobreajuste severo en esta fase. Sin embargo, la exactitud de validación se estabiliza alrededor de 0.89, lo que sugiere que el modelo alcanza una meseta de desempeño bajo esta configuración inicial.



### 8.2 Entrenamiento de ResNet50

En esta fase se entrena solamente la cabeza clasificadora de ResNet50, manteniendo congelado el extractor convolucional preentrenado en ImageNet.



In [ ]:

resnet_optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
resnet_model.compile(
    optimizer=resnet_optimizer,
    loss=resnet_criterion,
    metrics=["accuracy"],
)

resnet_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=BEST_RESNET_PATH,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
)

print("Configuración de entrenamiento")
print(f"- Modelo: ResNet50")
print(f"- Épocas: {NUM_EPOCHS}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Learning rate: {LEARNING_RATE}")
print(f"- Optimizador: {resnet_optimizer.__class__.__name__}")

resnet_history = resnet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=NUM_EPOCHS,
    callbacks=[resnet_checkpoint_callback],
    verbose=1,
)

resnet_model.load_weights(BEST_RESNET_PATH)
resnet_best_val_loss = min(resnet_history.history["val_loss"])
resnet_epochs = range(1, len(resnet_history.history["loss"]) + 1)

print(f"\nMejor pérdida de validación ResNet50: {resnet_best_val_loss:.4f}")
print(f"Pesos guardados en: {BEST_RESNET_PATH}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Curvas de aprendizaje - ResNet50", fontsize=12, fontweight="bold")

axes[0].plot(resnet_epochs, resnet_history.history["loss"], marker="o", label="Train")
axes[0].plot(resnet_epochs, resnet_history.history["val_loss"], marker="o", label="Validation")
axes[0].set_title("Pérdida por época")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(resnet_epochs, resnet_history.history["accuracy"], marker="o", label="Train")
axes[1].plot(resnet_epochs, resnet_history.history["val_accuracy"], marker="o", label="Validation")
axes[1].set_title("Accuracy por época")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()




#### 8.2.1 Conclusiones preliminares

El entrenamiento inicial de ResNet50, con el extractor convolucional congelado, superó el desempeño de EfficientNet-B0 en validación. La mejor pérdida de validación fue 0.1999, obtenida al final de las 10 épocas, con val_accuracy = 0.9197. La mayor exactitud de validación se observó en la época 7, con val_accuracy = 0.9303.

Aunque las curvas muestran algunas fluctuaciones en validación, especialmente en las épocas 6 y 9, la tendencia general indica que la capa clasificadora logró aprovechar de forma efectiva las características preentrenadas de la red residual. Esta fase establece una base más sólida para el ajuste fino posterior.



### 8.3 Fine-tuning de ResNet50

Después de entrenar la cabeza clasificadora, se descongelan los bloques superiores de ResNet50. El fine-tuning se realiza con una tasa de aprendizaje menor para ajustar las capas profundas sin modificar de forma agresiva las representaciones preentrenadas.



In [ ]:

resnet_model.load_weights(BEST_RESNET_PATH)
resnet_base_model.trainable = True

for layer in resnet_base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = layer.name.startswith("conv5_")

resnet_ft_optimizer = tf.keras.optimizers.Adam(learning_rate=RESNET_FINE_TUNE_LR)
resnet_model.compile(
    optimizer=resnet_ft_optimizer,
    loss=resnet_criterion,
    metrics=["accuracy"],
)

resnet_ft_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=BEST_RESNET_FT_PATH,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
)

print("Configuración de fine-tuning")
print(f"- Learning rate: {RESNET_FINE_TUNE_LR}")
print(f"- Épocas: {NUM_FINE_TUNE_EPOCHS}")
print(f"- Parámetros entrenables: {sum(np.prod(variable.shape) for variable in resnet_model.trainable_variables):,}")

resnet_ft_history = resnet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=NUM_FINE_TUNE_EPOCHS,
    callbacks=[resnet_ft_checkpoint_callback],
    verbose=1,
)

resnet_model.load_weights(BEST_RESNET_FT_PATH)
resnet_ft_best_val_loss = min(resnet_ft_history.history["val_loss"])
resnet_ft_epochs = range(1, len(resnet_ft_history.history["loss"]) + 1)

print(f"\nMejor pérdida de validación ResNet50 + fine-tuning: {resnet_ft_best_val_loss:.4f}")
print(f"Pesos guardados en: {BEST_RESNET_FT_PATH}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Curvas de aprendizaje - ResNet50 + Fine-tuning", fontsize=12, fontweight="bold")

axes[0].plot(resnet_ft_epochs, resnet_ft_history.history["loss"], marker="o", label="Train")
axes[0].plot(resnet_ft_epochs, resnet_ft_history.history["val_loss"], marker="o", label="Validation")
axes[0].set_title("Pérdida por época")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(resnet_ft_epochs, resnet_ft_history.history["accuracy"], marker="o", label="Train")
axes[1].plot(resnet_ft_epochs, resnet_ft_history.history["val_accuracy"], marker="o", label="Validation")
axes[1].set_title("Accuracy por época")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

combined_epochs = range(1, NUM_EPOCHS + NUM_FINE_TUNE_EPOCHS + 1)
combined_train_loss = resnet_history.history["loss"] + resnet_ft_history.history["loss"]
combined_val_loss = resnet_history.history["val_loss"] + resnet_ft_history.history["val_loss"]
combined_train_acc = resnet_history.history["accuracy"] + resnet_ft_history.history["accuracy"]
combined_val_acc = resnet_history.history["val_accuracy"] + resnet_ft_history.history["val_accuracy"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparación Transfer Learning + Fine-tuning - ResNet50", fontsize=14, fontweight="bold")

axes[0].plot(combined_epochs, combined_train_loss, label="Train Loss", linewidth=2)
axes[0].plot(combined_epochs, combined_val_loss, label="Validation Loss", linewidth=2)
axes[0].axvline(x=NUM_EPOCHS, color="red", linestyle="--", alpha=0.6, label="Inicio Fine-tuning")
axes[0].set_title("Pérdida total")
axes[0].set_xlabel("Épocas")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.2)

axes[1].plot(combined_epochs, combined_train_acc, label="Train Accuracy", linewidth=2)
axes[1].plot(combined_epochs, combined_val_acc, label="Validation Accuracy", linewidth=2)
axes[1].axvline(x=NUM_EPOCHS, color="red", linestyle="--", alpha=0.6, label="Inicio Fine-tuning")
axes[1].set_title("Exactitud total")
axes[1].set_xlabel("Épocas")
axes[1].set_ylabel("Accuracy")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()




#### 8.3.1 Conclusiones preliminares

El fine-tuning de ResNet50 produjo una mejora clara frente al entrenamiento con el extractor congelado. La pérdida de validación bajó hasta 0.0616 en la época 7, con val_accuracy = 0.9742. Aunque en las épocas posteriores la exactitud se mantuvo alta, la pérdida de validación aumentó ligeramente, por lo que la selección del mejor modelo por val_loss resulta adecuada.

La diferencia entre entrenamiento y validación al final de la fase (accuracy = 0.9917 frente a val_accuracy = 0.9727) sugiere un ajuste fuerte del modelo, pero no un deterioro severo de la generalización. Aun así, la época 7 parece representar el mejor balance entre desempeño y estabilidad en validación.



## 9. Evaluación y análisis de resultados



In [ ]:

def plot_classification_metrics(report, title):
    precision_scores = [report[class_name]["precision"] for class_name in class_names]
    recall_scores = [report[class_name]["recall"] for class_name in class_names]
    f1_scores = [report[class_name]["f1-score"] for class_name in class_names]
    positions = np.arange(len(class_names))
    bar_width = 0.25

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(positions - bar_width, precision_scores, width=bar_width, label="Precision")
    ax.bar(positions, recall_scores, width=bar_width, label="Recall")
    ax.bar(positions + bar_width, f1_scores, width=bar_width, label="F1-score")
    ax.set_title(title)
    ax.set_xlabel("Clase")
    ax.set_ylabel("Score")
    ax.set_xticks(positions)
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(cm, title):
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta real")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    threshold = cm.max() / 2
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(
                col,
                row,
                cm[row, col],
                ha="center",
                va="center",
                color="white" if cm[row, col] > threshold else "black",
            )

    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()


def plot_qualitative_examples(y_true, y_pred, title):
    correct_indices = [index for index, (true, pred) in enumerate(zip(y_true, y_pred)) if true == pred][:4]
    incorrect_indices = [index for index, (true, pred) in enumerate(zip(y_true, y_pred)) if true != pred][:4]

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle(title, fontsize=14)

    for ax in axes.flat:
        ax.axis("off")

    for column, index in enumerate(correct_indices):
        axes[0, column].imshow(plt.imread(test_paths[index]), cmap="gray")
        axes[0, column].set_title(
            f"Correcta\nReal: {class_names[y_true[index]]}\nPred: {class_names[y_pred[index]]}"
        )

    for column, index in enumerate(incorrect_indices):
        axes[1, column].imshow(plt.imread(test_paths[index]), cmap="gray")
        axes[1, column].set_title(
            f"Incorrecta\nReal: {class_names[y_true[index]]}\nPred: {class_names[y_pred[index]]}"
        )

    plt.tight_layout()
    plt.show()


def evaluate_model(model, dataset, targets, title):
    test_loss, test_accuracy = model.evaluate(dataset, verbose=0)
    test_probabilities = model.predict(dataset, verbose=0)
    y_true = np.array(targets)
    y_pred = np.argmax(test_probabilities, axis=1)

    print(f"Resultados en test - {title}")
    print(f"- Loss: {test_loss:.4f}")
    print(f"- Accuracy: {test_accuracy:.4f}")

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        output_dict=True,
    )

    print("\nReporte por clase")
    for class_name in class_names:
        metrics = report[class_name]
        print(
            f"- {class_name}: precision={metrics['precision']:.4f} | "
            f"recall={metrics['recall']:.4f} | "
            f"f1-score={metrics['f1-score']:.4f} | "
            f"support={int(metrics['support'])}"
        )

    print(
        f"- Macro avg: precision={report['macro avg']['precision']:.4f} | "
        f"recall={report['macro avg']['recall']:.4f} | "
        f"f1-score={report['macro avg']['f1-score']:.4f}"
    )

    plot_classification_metrics(report, f"Desempeño por clase - {title}")
    cm = confusion_matrix(y_true, y_pred)
    plot_confusion_matrix(cm, f"Matriz de confusión - {title}")
    plot_qualitative_examples(y_true, y_pred, f"Ejemplos correctamente e incorrectamente clasificados - {title}")

    return {
        "model": title,
        "loss": test_loss,
        "accuracy": test_accuracy,
        "report": report,
        "cm": cm,
        "y_true": y_true,
        "y_pred": y_pred,
    }




### 9.1 Evaluación de EfficientNet-B0



In [ ]:

efficientnet_results = evaluate_model(
    efficientnet_model,
    test_dataset,
    test_targets,
    "EfficientNet-B0",
)




#### 9.1.1 Conclusiones preliminares

En el conjunto de prueba, EfficientNet-B0 obtuvo una exactitud de 0.8924, con promedio macro de precision = 0.8907, recall = 0.8879 y f1-score = 0.8878. El resultado confirma un desempeño general satisfactorio, especialmente en las clases healthy y pituitary, cuyos f1-score fueron 0.9647 y 0.9189, respectivamente.

La clase con mayor dificultad fue meningioma, con recall = 0.7451 y f1-score = 0.7889. Esto indica que una proporción relevante de imágenes de esta clase fue clasificada como otra categoría. Este comportamiento sugiere que las características visuales de meningioma pueden solaparse con las de otras clases, por lo que conviene analizar sus errores con mayor detalle.



### 9.2 Evaluación de ResNet50 y ResNet50 con fine-tuning



In [ ]:

resnet_model.load_weights(BEST_RESNET_PATH)
resnet_results = evaluate_model(
    resnet_model,
    test_dataset,
    test_targets,
    "ResNet50",
)

resnet_model.load_weights(BEST_RESNET_FT_PATH)
resnet_ft_results = evaluate_model(
    resnet_model,
    test_dataset,
    test_targets,
    "ResNet50 + Fine-tuning",
)




#### 9.2.1 Conclusiones preliminares

En prueba, ResNet50 obtuvo una accuracy de 0.9273, con promedio macro de precision = 0.9267, recall = 0.9251 y f1-score = 0.9253. Este resultado mejora el desempeño de EfficientNet-B0, aunque la clase meningioma continuó siendo la más difícil, con f1-score = 0.8508.

Después del fine-tuning, ResNet50 alcanzó una accuracy de 0.9667, con promedio macro de precision = 0.9662, recall = 0.9660 y f1-score = 0.9657. La mejora fue especialmente relevante en meningioma, cuyo recall subió a 0.9608 y cuyo f1-score llegó a 0.9333. Esto indica que ajustar los bloques superiores permitió capturar características más específicas del dominio de MRI.



### 9.3 Comparación de enfoques



In [ ]:

comparison_rows = [
    ("EfficientNet-B0", efficientnet_results["loss"], efficientnet_results["accuracy"]),
    ("ResNet50", resnet_results["loss"], resnet_results["accuracy"]),
    ("ResNet50 + Fine-tuning", resnet_ft_results["loss"], resnet_ft_results["accuracy"]),
]

print(f"{'Modelo':<28} {'Loss':>10} {'Accuracy':>10}")
print("-" * 52)
for model_name, loss, accuracy in comparison_rows:
    print(f"{model_name:<28} {loss:>10.4f} {accuracy:>10.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    [row[0] for row in comparison_rows],
    [row[2] for row in comparison_rows],
    color=["#355070", "#6d597a", "#b56576"],
)
ax.set_title("Comparación de accuracy en test")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()




#### 9.3.1 Conclusiones preliminares

La comparación global muestra una mejora progresiva entre los enfoques evaluados. EfficientNet-B0 alcanzó accuracy = 0.8924, ResNet50 aumentó a 0.9273 y ResNet50 + fine-tuning llegó a 0.9667. En términos relativos, el ajuste fino mejoró 3.94 puntos porcentuales frente a ResNet50 sin fine-tuning y 7.43 puntos porcentuales frente a EfficientNet-B0.

La reducción de la pérdida también fue consistente: 0.2885 en EfficientNet-B0, 0.2049 en ResNet50 y 0.1172 en ResNet50 + fine-tuning. Por tanto, el mejor enfoque para este experimento fue ResNet50 + fine-tuning, tanto por exactitud como por equilibrio entre precision, recall y f1-score.



## 10. Conclusiones

En este microproyecto se desarrolló un pipeline completo para la clasificación de imágenes de resonancia magnética cerebral en cuatro categorías: glioma, healthy, meningioma y pituitary. A partir del preprocesamiento del dataset, la eliminación de duplicados exactos, la construcción de los conjuntos de entrenamiento, validación y prueba, y el uso de redes convolucionales basadas en transferencia de aprendizaje, fue posible obtener un desempeño satisfactorio en la tarea planteada.

En términos cuantitativos, EfficientNet-B0 alcanzó una accuracy de 89.24% en el conjunto de prueba, con promedio macro de precision = 0.8907, recall = 0.8879 y f1-score = 0.8878. Por su parte, ResNet50 obtuvo una accuracy de 92.73%, con promedio macro de precision = 0.9267, recall = 0.9251 y f1-score = 0.9253. El mejor resultado se obtuvo con ResNet50 + fine-tuning, que alcanzó una accuracy de 96.67%, con promedio macro de precision = 0.9662, recall = 0.9660 y f1-score = 0.9657.

Los resultados muestran una mejora progresiva entre los tres enfoques evaluados. La pérdida en prueba disminuyó de 0.2885 con EfficientNet-B0, a 0.2049 con ResNet50, y finalmente a 0.1172 con ResNet50 + fine-tuning. Esto indica que la arquitectura residual tuvo una mejor capacidad de adaptación al dataset y que el ajuste fino de los bloques superiores permitió especializar mejor las representaciones visuales para las imágenes de MRI.

El análisis por clase evidencia que healthy y pituitary fueron las categorías más consistentes en los tres enfoques. En cambio, meningioma fue la clase más difícil para EfficientNet-B0 y para ResNet50 sin fine-tuning. Con el ajuste fino, el desempeño de esta clase mejoró de manera importante, alcanzando recall = 0.9608 y f1-score = 0.9333, aunque siguió siendo la categoría con el f1-score más bajo dentro del mejor modelo.

Se eligió EfficientNet-B0 porque ofrece una arquitectura eficiente, capaz de extraer características visuales relevantes con un menor costo computacional que otras redes más pesadas. También se evaluó ResNet50 porque sus conexiones residuales facilitan el entrenamiento de redes profundas y permiten ajustar representaciones de alto nivel mediante fine-tuning. La comparación entre ambas arquitecturas permitió observar que el fine-tuning de ResNet50 generó el mejor balance entre precisión, recall y f1-score.

Aunque el desempeño alcanzado por ResNet50 + fine-tuning es alto, en un contexto clínico real sería necesario validar el modelo con un conjunto externo e idealmente con una partición a nivel de paciente. Esto es importante porque, si existen imágenes similares o cortes relacionados de un mismo paciente en distintos conjuntos, las métricas podrían sobreestimar la capacidad de generalización. Además, los falsos negativos siguen siendo especialmente críticos en aplicaciones médicas, ya que podrían retrasar el diagnóstico.

Como trabajo futuro, sería conveniente validar el modelo con datos externos, analizar con mayor detalle los errores por clase y aplicar técnicas de interpretabilidad visual para entender mejor las decisiones del modelo. También podría explorarse el uso de early stopping y nuevas configuraciones de fine-tuning para mejorar las clases más difíciles sin aumentar el sobreajuste.



## 11. Referencias

- Dataset utilizado: Rajarshi Mandal. Brain Tumor (MRI Scans). Kaggle. Disponible en: https://www.kaggle.com/datasets/rm1000/brain-tumor-mri-scans
- Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks. ICML 2019. Disponible en: https://research.google/pubs/efficientnet-rethinking-model-scaling-for-convolutional-neural-networks/
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep Residual Learning for Image Recognition. CVPR 2016. Disponible en: https://arxiv.org/abs/1512.03385

